# Módulo 0 — Análisis climático histórico: Jocolí, Lavalle, Mendoza

Este módulo presenta el análisis de los datos climáticos históricos del período 1990–2024 para la localidad de Jocolí, departamento de Lavalle, provincia de Mendoza. Los datos se obtienen de la API de reanálisis ERA5-Land de Open-Meteo, con resolución temporal horaria y resolución espacial de aproximadamente 9 km.

El objetivo es caracterizar el régimen térmico e hídrico del sitio y evaluar su aptitud para el cultivo de pistacho (*Pistacia vera* L.), con especial atención a las variables que condicionan el rendimiento: acumulación de frío invernal, calor estival y déficit hídrico.

**Variables analizadas:** temperatura del aire a 2 m · precipitación · evapotranspiración de referencia (ET₀, método FAO Penman-Monteith)  
**Período:** 1990–2024 (35 años)  
**Fuente:** Open-Meteo Archive API — ERA5-Land (ECMWF)

In [1]:
import subprocess
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    REPO_URL = "https://github.com/Emilialandgrebe/tesis-maestria.git"
    ROOT = Path("/content/tesis-maestria")
    if not ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt"), "-q"],
        check=True,
    )
else:
    ROOT = Path().resolve().parent  # notebooks/ -> raiz del proyecto

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Entorno : {'Google Colab' if EN_COLAB else 'local'}")
print(f"ROOT    : {ROOT}")

Entorno : Google Colab
ROOT    : /content/tesis-maestria


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymannkendall as mk
from scipy.stats import theilslopes
from statsmodels.nonparametric.smoothers_lowess import lowess

from src.data.climate_fetcher import fetch_climate_data
from src.data.climate_features import (
    calcular_calor_verano,
    calcular_deficit_hidrico,
    calcular_heladas_tardias,
    calcular_horas_frio,
)

np.random.seed(42)

df = fetch_climate_data()

horas_frio      = calcular_horas_frio(df)
heladas         = calcular_heladas_tardias(df)
calor_verano    = calcular_calor_verano(df)
deficit_hidrico = calcular_deficit_hidrico(df)

In [ ]:
print("=" * 60)
print("RANGO DE FECHAS")
print("=" * 60)
print(f"Inicio : {df.index.min()}")
print(f"Fin    : {df.index.max()}")
print(f"Total  : {len(df):,} registros horarios")
print(f"         ({len(df) / 8760:.1f} años aprox.)")

In [ ]:
print("PRIMERAS FILAS")
df.head()

In [ ]:
print("ÚLTIMAS FILAS")
df.tail()

In [ ]:
print("ESTRUCTURA DEL DATAFRAME")
df.info()

In [ ]:
print("ESTADÍSTICAS DESCRIPTIVAS")
df.describe()

In [ ]:
print("=" * 60)
print("VALORES NULOS POR COLUMNA")
print("=" * 60)
nulos = df.isnull().sum()
pct   = (nulos / len(df) * 100).round(3)
resumen_nulos = nulos.to_frame("cantidad").assign(porcentaje=pct)
print(resumen_nulos)
print()
if nulos.sum() == 0:
    print("Sin valores nulos. Dataset completo.")
else:
    print(f"ATENCIÓN: {nulos.sum()} valores nulos en total.")

## Temperatura máxima diaria media en verano (enero–febrero)

Se calcula la temperatura máxima diaria media durante los meses de enero y febrero de cada año del período analizado. Estos meses corresponden a la etapa de mayor demanda térmica del pistacho, cuando el cultivo requiere temperaturas elevadas para completar el llenado y la apertura de cáscara del fruto. El rango óptimo para esta etapa es de 35 a 38 °C (Crane y Takeda, 1979; Ferguson, 2006).

Para el análisis de tendencia se aplica el **test de Mann-Kendall modificado** (Hamed y Rao, 1998), que es el método estándar recomendado por la Organización Meteorológica Mundial (OMM) para series temporales climáticas. A diferencia de la regresión lineal ordinaria, este test no asume distribución normal de los residuos ni independencia entre observaciones, lo que lo hace adecuado para series con autocorrelación. La magnitud de la tendencia se estima mediante el **estimador de Sen** (*Sen's Slope*), que calcula la mediana de todas las pendientes posibles entre pares de puntos y resulta robusto frente a valores atípicos. De manera complementaria, se aplica un suavizado LOWESS (*Locally Weighted Scatterplot Smoothing*) con fracción de 0,25 para capturar la estructura no lineal de la variabilidad interanual.

In [ ]:
años = calor_verano.index.values.astype(float)
temperaturas = calor_verano.values

# Test de Mann-Kendall con corrección de autocorrelación (Hamed & Rao, 1998)
mk_result = mk.hamed_rao_modification_test(temperaturas)

# Estimador de Sen (pendiente robusta)
sen = theilslopes(temperaturas, años)
linea_sen = sen.intercept + sen.slope * años

# Suavizado LOWESS
suavizado = lowess(temperaturas, años, frac=0.25)

# Banda ±1 desvío estándar
desvio    = calor_verano.std()
banda_sup = temperaturas + desvio
banda_inf = temperaturas - desvio

umbral_extremo = 37
años_extremos  = calor_verano[calor_verano >= umbral_extremo]

# Paleta
C_SERIE     = "#2C5F8A"
C_TENDENCIA = "#404040"
C_LOWESS    = "#1A1A1A"
C_EXTREMO   = "#8B2020"
C_OPTIMO    = "#4A6741"

fig, ax = plt.subplots(figsize=(13, 6))

ax.plot(años, temperaturas, color=C_SERIE, linewidth=2,
        marker="o", markersize=4, label="Tmax media ene–feb")

ax.fill_between(años, banda_inf, banda_sup, color=C_SERIE,
                alpha=0.12, label="±1 desvío estándar")

ax.fill_between(años, 35, 38, color=C_OPTIMO,
                alpha=0.10, label="Rango óptimo pistacho (35–38 °C)")
ax.axhline(35, color=C_OPTIMO, linewidth=0.8, linestyle="--")
ax.axhline(38, color=C_OPTIMO, linewidth=0.8, linestyle="--")

ax.plot(años, linea_sen, color=C_TENDENCIA, linestyle="--", linewidth=1.8,
        label=(f"Sen's Slope = {sen.slope:.3f} °C/año  "
               f"[IC 95%: {sen.low_slope:.3f}–{sen.high_slope:.3f}]\n"
               f"Mann-Kendall τ={mk_result.Tau:.2f}   p={mk_result.p:.3f}   "
               f"tendencia: {mk_result.trend}"))

ax.plot(suavizado[:, 0], suavizado[:, 1], color=C_LOWESS,
        linewidth=2, label="Suavizado LOWESS")

ax.axvspan(2015, 2024, color="#AAAAAA", alpha=0.10, label="Última década")

ax.scatter(años_extremos.index, años_extremos.values, color=C_EXTREMO,
           s=55, zorder=10, label=f"Años ≥ {umbral_extremo} °C")

ax.set_title(
    "Temperatura máxima diaria media del verano (enero–febrero)\n"
    "Jocolí, Lavalle, Mendoza (1990–2024)",
    fontsize=13, fontweight="bold", loc="center",
)
ax.set_xlabel("Año", fontsize=11)
ax.set_ylabel("Temperatura (°C)", fontsize=11)
ax.grid(axis="y", linestyle=":", alpha=0.4, color="#AAAAAA")
ax.legend(loc="upper left", fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.figtext(0.99, 0.01, "Fuente: Elaboración propia a partir de datos ERA5-Land (Open-Meteo).",
            ha="right", fontsize=8, color="#555555")
plt.show()

print("─" * 52)
print("RESULTADOS DEL ANÁLISIS DE TENDENCIA")
print("─" * 52)
print(f"Test de Mann-Kendall (Hamed & Rao, 1998)")
print(f"  Tendencia detectada : {mk_result.trend}")
print(f"  Kendall's τ         : {mk_result.Tau:.4f}")
print(f"  p-valor             : {mk_result.p:.4f}")
print(f"  Estadístico z       : {mk_result.z:.4f}")
print()
print(f"Estimador de Sen")
print(f"  Pendiente           : {sen.slope:.4f} °C/año")
print(f"  IC 95%              : [{sen.low_slope:.4f}, {sen.high_slope:.4f}]")
print(f"  Intercepto          : {sen.intercept:.4f}")


## Frecuencia de días con temperatura extrema (enero–febrero)

Se cuantifica el número de días por año en que la temperatura máxima diaria supera los umbrales de 35 °C, 38 °C y 40 °C durante el período estival (enero–febrero). Este indicador resulta agronómicamente más informativo que la temperatura media, ya que permite evaluar la intensidad y acumulación del estrés térmico al que se expone el cultivo durante la etapa crítica de maduración.

Valores superiores a 38 °C marcan el límite del rango óptimo para el pistacho, mientras que temperaturas por encima de los 40 °C se asocian a daño directo en el fruto, reducción del calibre comercial y mayor incidencia de frutos vacíos (blank nuts).

In [ ]:
mascara_verano = df.index.month.isin([1, 2])
tmax_diaria_verano = (
    df.loc[mascara_verano, "temperature_2m"]
    .groupby(df.index[mascara_verano].date)
    .max()
)
tmax_diaria_verano.index = pd.to_datetime(tmax_diaria_verano.index)

dias35 = tmax_diaria_verano.groupby(tmax_diaria_verano.index.year).apply(lambda x: (x > 35).sum())
dias38 = tmax_diaria_verano.groupby(tmax_diaria_verano.index.year).apply(lambda x: (x > 38).sum())
dias40 = tmax_diaria_verano.groupby(tmax_diaria_verano.index.year).apply(lambda x: (x > 40).sum())

# Paleta sobria: gradiente cálido según intensidad del umbral
C_35 = "#D9A86C"
C_38 = "#B05C2E"
C_40 = "#7A1C1C"

fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(dias35.index, dias35.values, color=C_35, linewidth=2.2,
        marker="o", markersize=4, label=">35 °C")
ax.plot(dias38.index, dias38.values, color=C_38, linewidth=2.2,
        marker="o", markersize=4, label=">38 °C")
ax.plot(dias40.index, dias40.values, color=C_40, linewidth=2.2,
        marker="o", markersize=4, label=">40 °C")

ax.set_title(
    "Frecuencia anual de días con temperatura extrema (enero–febrero)\n"
    "Jocolí, Lavalle, Mendoza (1990–2024)",
    fontsize=13, fontweight="bold", loc="center",
)
ax.set_ylabel("Número de días", fontsize=11)
ax.set_xlabel("Año", fontsize=11)
ax.grid(alpha=0.3, linestyle=":", color="#AAAAAA")
ax.legend(fontsize=10, framealpha=0.9)
plt.tight_layout()
plt.figtext(0.99, 0.01, "Fuente: Elaboración propia a partir de datos ERA5-Land (Open-Meteo).",
            ha="right", fontsize=8, color="#555555")
plt.show()


## Distribución de la temperatura máxima diaria por década

Se analiza la distribución estadística de las temperaturas máximas diarias de enero–febrero agrupadas por décadas. Este enfoque permite visualizar no solo el desplazamiento de la mediana a lo largo del tiempo, sino también los cambios en la variabilidad interna y en los valores extremos, información que la temperatura media anual por sí sola no refleja.

Cada caja muestra el rango intercuartílico (P25–P75), la línea central indica la mediana y los puntos exteriores corresponden a valores atípicos según el criterio de 1,5 × IQR. Las líneas horizontales de referencia señalan los límites del rango óptimo de temperatura para el cultivo (35 y 38 °C).

In [ ]:
decadas_etiquetas = {
    1990: "1990–1999",
    2000: "2000–2009",
    2010: "2010–2019",
    2020: "2020–2024",
}
# Gradiente de azul frío a rojo cálido por década
colores_decadas = {
    1990: "#7BA7C0",
    2000: "#4E7FA8",
    2010: "#B87448",
    2020: "#8B2020",
}

decada_serie = (tmax_diaria_verano.index.year // 10) * 10

datos_boxplot = [
    tmax_diaria_verano[decada_serie == d].values
    for d in sorted(decadas_etiquetas)
]
etiquetas     = [decadas_etiquetas[d] for d in sorted(decadas_etiquetas)]
colores_lista = [colores_decadas[d]   for d in sorted(decadas_etiquetas)]

fig, ax = plt.subplots(figsize=(10, 6))

bp = ax.boxplot(
    datos_boxplot,
    labels=etiquetas,
    patch_artist=True,
    widths=0.5,
    medianprops=dict(color="black", linewidth=2),
    whiskerprops=dict(linewidth=1.4, color="#444444"),
    capprops=dict(linewidth=1.4, color="#444444"),
    flierprops=dict(marker="o", markersize=3.5, linestyle="none",
                    alpha=0.5, markerfacecolor="#666666", markeredgecolor="none"),
)
for patch, color in zip(bp["boxes"], colores_lista):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

C_OPTIMO = "#4A6741"
ax.axhline(35, color=C_OPTIMO, linewidth=1, linestyle="--",
           label="35 °C (límite inferior rango óptimo)")
ax.axhline(38, color=C_OPTIMO, linewidth=1, linestyle="-.",
           label="38 °C (límite superior rango óptimo)")

ax.set_title(
    "Distribución de la temperatura máxima diaria en verano (ene–feb) por década\n"
    "Jocolí, Lavalle, Mendoza (1990–2024)",
    fontsize=13, fontweight="bold", loc="center",
)
ax.set_ylabel("Temperatura máxima diaria (°C)", fontsize=11)
ax.set_xlabel("Década", fontsize=11)
ax.grid(axis="y", linestyle=":", alpha=0.4, color="#AAAAAA")
ax.legend(fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.figtext(0.99, 0.01, "Fuente: Elaboración propia a partir de datos ERA5-Land (Open-Meteo).",
            ha="right", fontsize=8, color="#555555")
plt.show()


## Anomalías térmicas respecto al período de referencia 1990–2020

Se calculan las anomalías de temperatura máxima media de verano tomando como período de referencia los años 1990–2020, criterio estándar establecido por la Organización Meteorológica Mundial (OMM) para la caracterización climatológica. La anomalía de cada año se obtiene como la diferencia entre el valor observado y la media del período de referencia.

Las barras positivas indican años más cálidos que el promedio histórico y las barras negativas corresponden a años más frescos. Se ajusta además una tendencia lineal sobre las anomalías para cuantificar el ritmo de calentamiento en el período analizado, expresado en °C por año.

In [ ]:
ref_media   = calor_verano.loc[1990:2020].mean()
anomalias   = calor_verano - ref_media
colores_bar = ["#B5432A" if v >= 0 else "#4472C4" for v in anomalias]

# Mann-Kendall y Sen's Slope sobre las anomalías
mk_an  = mk.hamed_rao_modification_test(anomalias.values)
sen_an = theilslopes(anomalias.values, anomalias.index.astype(float))
linea_sen_an = sen_an.intercept + sen_an.slope * anomalias.index.astype(float)

fig, ax = plt.subplots(figsize=(13, 5))

ax.bar(anomalias.index, anomalias.values, color=colores_bar, width=0.8, alpha=0.85)
ax.axhline(0, color="#222222", linewidth=0.9)

ax.plot(
    anomalias.index, linea_sen_an,
    color="#404040", linestyle="--", linewidth=1.8,
    label=(f"Sen's Slope = {sen_an.slope:.3f} °C/año  "
           f"| MK τ={mk_an.Tau:.2f}   p={mk_an.p:.3f}   {mk_an.trend}"),
)

ax.set_title(
    "Anomalías de temperatura máxima media en verano respecto al período base 1990–2020\n"
    "Jocolí, Lavalle, Mendoza",
    fontsize=13, fontweight="bold", loc="center",
)
ax.set_xlabel("Año", fontsize=11)
ax.set_ylabel("Anomalía (°C)", fontsize=11)
ax.grid(axis="y", linestyle=":", alpha=0.4, color="#AAAAAA")
ax.legend(fontsize=10, framealpha=0.9)
plt.tight_layout()
plt.figtext(0.99, 0.01, "Fuente: Elaboración propia a partir de datos ERA5-Land (Open-Meteo).",
            ha="right", fontsize=8, color="#555555")
plt.show()

print(f"\nMedia de referencia 1990–2020 : {ref_media:.2f} °C")
print(f"Anomalía promedio 2021–2024   : {anomalias.loc[2021:].mean():+.2f} °C")


## Déficit hídrico anual (ET₀ − Precipitación)

Se estima el déficit hídrico anual como la diferencia entre la evapotranspiración de referencia (ET₀) y la precipitación total acumulada en el año, ambas expresadas en milímetros. Este indicador representa la demanda neta de agua de riego necesaria para cubrir la evapotranspiración del cultivo, bajo el supuesto de que no se considera el aporte de agua freática ni las pérdidas por eficiencia del sistema de riego.

La zona de Jocolí se localiza en el piedemonte árido mendocino, donde la precipitación media anual no supera los 150 mm, de modo que la práctica totalidad de la demanda hídrica del cultivo debe cubrirse mediante riego superficial o por goteo. Se incluye una línea de referencia en 600 mm, valor que corresponde al aporte mínimo de riego contemplado en el plan de negocios del proyecto analizado.

In [ ]:
años_dh = deficit_hidrico.index
tendencia_dh = np.polyfit(años_dh, deficit_hidrico.values, 1)
linea_tendencia_dh = np.poly1d(tendencia_dh)

fig, ax = plt.subplots(figsize=(13, 5))

ax.bar(años_dh, deficit_hidrico.values, color="#4472C4", alpha=0.80,
       label="Déficit hídrico anual (mm)")

ax.axhline(600, color="#8B2020", linewidth=1.8, linestyle="--",
           label="600 mm (mínimo de riego — plan de negocios)")

ax.plot(años_dh, linea_tendencia_dh(años_dh), color="#404040", linewidth=1.8,
        linestyle="-.", label=f"Tendencia ({tendencia_dh[0]:+.1f} mm/año)")

ax.set_title(
    "Déficit hídrico anual (ET₀ − Precipitación)\n"
    "Jocolí, Lavalle, Mendoza (1990–2024)",
    fontsize=13, fontweight="bold", loc="center",
)
ax.set_xlabel("Año", fontsize=11)
ax.set_ylabel("mm / año", fontsize=11)
ax.grid(axis="y", linestyle=":", alpha=0.4, color="#AAAAAA")
ax.legend(fontsize=10, framealpha=0.9)
plt.tight_layout()
plt.figtext(0.99, 0.01, "Fuente: Elaboración propia a partir de datos ERA5-Land (Open-Meteo).",
            ha="right", fontsize=8, color="#555555")
plt.show()


## Parámetros calibrados — exportar al Módulo 1

In [ ]:
from scipy import stats

UMBRAL_FRIO_CRITICO = 800

_dist, _params = "norm", stats.norm.fit(horas_frio.values)

PARAMS_CLIMA_JOCOLI = {
    "horas_frio_media":          float(horas_frio.mean()),
    "horas_frio_std":            float(horas_frio.std()),
    "horas_frio_dist":           _dist,
    "horas_frio_dist_params":    _params,
    "prob_anio_critico":         float((horas_frio < UMBRAL_FRIO_CRITICO).mean()),
    "prob_helada_tardia":        float((heladas > 0).mean()),
    "calor_verano_media":        float(calor_verano.mean()),
    "deficit_hidrico_media_mm":  float(deficit_hidrico.mean()),
}

print("PARAMS_CLIMA_JOCOLI = {")
for k, v in PARAMS_CLIMA_JOCOLI.items():
    print(f"    {k!r}: {v},")
print("}")